# Binary Class Penguin Logistic Regression (PyTorch)

In [1]:
import numpy as np
import torch

In [2]:
# read the variables: class label and features

def string2float(item):
    return float(item) if item != '' else None

def read_strings(filename, col):
    with open(filename) as f:
        lines = f.readlines()
    values = [line.strip().split(',')[col] for line in lines[1:]]
    return values

def read_floats(filename, col):
    with open(filename) as f:
        lines = f.readlines()
    values = [string2float(line.strip().split(',')[col]) for line in lines[1:]]
    return values

def read_data_and_label(filename):
    x1 = np.array(read_floats(filename, col=4))  # flipper length
    x2 = np.array(read_floats(filename, col=5))  # body mass
    y = np.array(read_strings(filename, col=0))  # species

    y = [s == 'Gentoo' for s in y]
    y = np.array(y).astype(float)

    idx = (x1 != None) & (x2 != None) & (y != None)
    x1 = x1[idx]
    x2 = x2[idx]
    y = y[idx]

    X = np.stack((x1, x2), axis=1).astype(float)
    y = y.reshape((y.size, 1)).astype(float)

    return X, y

In [3]:
def min_max_normalize(X, min_val=None, max_val=None):
    X = np.array(X)
    if min_val is None:
        min_val = np.min(X, axis=0, keepdims=True)
    if max_val is None:
        max_val = np.max(X, axis=0, keepdims=True)
    range_val = max_val - min_val
    range_val = np.where(range_val == 0, 1, range_val)
    X_norm = (X - min_val) / range_val
    return X_norm, min_val, max_val

In [4]:
def z_score_normalize(X, mean=None, std=None):
    if mean is None:
        mean = np.mean(X, axis=0)
    if std is None:
        std = np.std(X, axis=0)
    std = np.where(std == 0, 1, std)
    X_norm = (X - mean) / std
    return X_norm, mean, std

In [5]:
def print_formatted_vector(vec):
    s = ' '.join(f"{x.item():.4f}" for x in vec)
    print(s)

In [6]:
class LogisticRegressionModel(torch.nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.linear = torch.nn.Linear(in_dim, out_dim)

    def forward(self, x):
        x = self.linear(x)
        x = torch.sigmoid(x)
        return x

In [7]:
# training data

filename = 'data/palmer-penguins/palmer-penguins-train.txt'
X, y = read_data_and_label(filename)

X, X_mean, X_std = z_score_normalize(X)

X = torch.from_numpy(X).float()
y = torch.from_numpy(y).float()

In [16]:
# init training model

torch.manual_seed(0)

model = LogisticRegressionModel(in_dim=X.shape[1], out_dim=1)

print(model.linear.weight)
print(model.linear.bias)

Parameter containing:
tensor([[-0.0053,  0.3793]], requires_grad=True)
Parameter containing:
tensor([-0.5820], requires_grad=True)


In [19]:
def criterion(y_pred, y):
    epsilon = 1e-15
    y_pred = torch.clip(y_pred, epsilon, 1 - epsilon)
    losses = -(y*torch.log(y_pred) + (1 - y)*torch.log(1 - y_pred))
    return torch.mean(losses)

# criterion = torch.nn.BCEWithLogitsLoss()

optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

In [20]:
# training

for epoch in range(1000):
    optimizer.zero_grad()
    y_pred = model(X)

    loss = criterion(y_pred, y)
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 200 == 0:
        print('epoch {}: loss {:.4f}'.format(epoch + 1, loss.item()))
        print_formatted_vector(y[::20])
        print_formatted_vector(y_pred[::20])

epoch 200: loss 0.1141
0.0000 0.0000 0.0000 0.0000 1.0000 1.0000 1.0000 0.0000 0.0000
0.0228 0.1654 0.0917 0.1436 0.9296 0.9330 0.9764 0.0734 0.0707
epoch 400: loss 0.0784
0.0000 0.0000 0.0000 0.0000 1.0000 1.0000 1.0000 0.0000 0.0000
0.0075 0.0973 0.0447 0.0868 0.9520 0.9568 0.9889 0.0359 0.0343
epoch 600: loss 0.0637
0.0000 0.0000 0.0000 0.0000 1.0000 1.0000 1.0000 0.0000 0.0000
0.0037 0.0686 0.0279 0.0614 0.9635 0.9680 0.9932 0.0222 0.0211
epoch 800: loss 0.0555
0.0000 0.0000 0.0000 0.0000 1.0000 1.0000 1.0000 0.0000 0.0000
0.0022 0.0528 0.0196 0.0469 0.9706 0.9746 0.9953 0.0154 0.0145
epoch 1000: loss 0.0501
0.0000 0.0000 0.0000 0.0000 1.0000 1.0000 1.0000 0.0000 0.0000
0.0014 0.0427 0.0148 0.0375 0.9755 0.9790 0.9965 0.0114 0.0107


In [21]:
print(model.linear.weight)
print(model.linear.bias)

Parameter containing:
tensor([[2.5037, 1.7303]], requires_grad=True)
Parameter containing:
tensor([-3.9192], requires_grad=True)


In [22]:
# test data

filename = 'data/palmer-penguins/palmer-penguins-test.txt'

X, y = read_data_and_label(filename)

X, _, _ = z_score_normalize(X, X_mean, X_std)

X = torch.from_numpy(X).float()
y = torch.from_numpy(y).float()

In [23]:
# evaluation

model.eval()

with torch.no_grad():
    y_pred = model(X)
    y_pred = (y_pred > 0.5).float()
    y_corr = y == y_pred
    print(torch.mean(y_corr.float()))  # accuarcy

tensor(0.9941)
